In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [217]:
print("loading cache data...")
D = p.load(open("cached_data/movie_lens_preprocessed.p", "rb"))

row = D["userId"]
col = D["movieId"]
data = D["rating"]

movies = D["movies"]

print("done")

loading cache data...
done


In [5]:
def find_match_using_terms(terms, movies=movies, case_insensitive=False):
    assert type(terms) in (list, tuple, set)
    
    title = movies.title
    if case_insensitive:
        title = title.str.lower()
    
    matches = True
    for term in terms:
        matches &= title.str.contains(term)
        
    matches = np.where(matches)[0]

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", movies.loc[matches, "title"].tolist())
        
    return matches[0]

In [6]:
mat = csr_matrix((data.astype(bool), (row, col))).astype(np.int64)

In [7]:
a = find_match_using_terms(["Sense and Sensibility", "1995"])
# a = find_match_using_terms(["Knives Out", "2019"])
# a = find_match_using_terms(["Witness", "1957"])

movies.loc[a]

title         Sense and Sensibility (1995)
genres                       Drama|Romance
imdbId                              114388
tmdbId                              4584.0
avg_rating                        7.891793
num_votes                            18677
Name: 16, dtype: object

In [8]:
b = find_match_using_terms(["Pride and Prejudice", "1995"])
# b = find_match_using_terms(["Death", "Nile", "1978"])
# b = find_match_using_terms(["Amadeus"])

movies.loc[b]

title         Pride and Prejudice (1995)
genres                     Drama|Romance
imdbId                            112130
tmdbId                          164721.0
avg_rating                       7.98947
num_votes                           2607
Name: 7382, dtype: object

In [33]:
a1 = find_match_using_terms(["Lord of the Rings", "Fellowship"])
a2 = find_match_using_terms(["Lord of the Rings", "Two"])
a3 = find_match_using_terms(["Lord of the Rings", "Return"])

In [34]:
b1 = find_match_using_terms(["Star Wars", "New Hope"])
b2 = find_match_using_terms(["Star Wars", "Empire Strikes Back"])
b3 = find_match_using_terms(["Star Wars", "Return of the Jedi"])
# a2 = find_match_using_terms(["Lord of the Rings", "Two"])
# a3 = find_match_using_terms(["Lord of the Rings", "Return"])

In [35]:
c1 = find_match_using_terms(["Sense and Sensibility", "1995"])
c2 = find_match_using_terms(["Pride and Prejudice", "1995"])

In [36]:
d1 = find_match_using_terms(["Matrix", "1999"])
d2 = find_match_using_terms(["Matrix", "Revolutions, The"])

In [37]:
e1 = find_match_using_terms(["Excellent Adventure"])
e2 = find_match_using_terms(["Bogus Journey"])

In [62]:
x = [a1, a2, a3, b1, b2, b3, c1, c2, d1, d2, e1, e2]

In [140]:
G = mat[:, x].T @ mat[:, x]
G = G.toarray().astype(np.float64)
np.fill_diagonal(G, 0)
G /= G.sum()

G

array([[0.00000000e+00, 2.64267288e-02, 2.63827746e-02, 1.81804330e-02,
        1.72671624e-02, 1.47124468e-02, 1.99356706e-03, 7.51616782e-04,
        2.29816965e-02, 4.67672664e-03, 2.17426765e-03, 6.43684807e-04],
       [2.64267288e-02, 0.00000000e+00, 2.56541117e-02, 1.68901331e-02,
        1.60760036e-02, 1.38577818e-02, 1.78942423e-03, 7.11081244e-04,
        2.12132726e-02, 4.62495837e-03, 1.97500862e-03, 5.85567590e-04],
       [2.63827746e-02, 2.56541117e-02, 0.00000000e+00, 1.71030667e-02,
        1.61868659e-02, 1.35974753e-02, 1.68832957e-03, 7.33546723e-04,
        2.19736802e-02, 4.65181926e-03, 1.84509955e-03, 5.46985572e-04],
       [1.81804330e-02, 1.68901331e-02, 1.71030667e-02, 0.00000000e+00,
        2.70977629e-02, 2.39965501e-02, 3.65845439e-03, 6.03637649e-04,
        2.30603257e-02, 3.80594515e-03, 2.30759538e-03, 6.52964027e-04],
       [1.72671624e-02, 1.60760036e-02, 1.61868659e-02, 2.70977629e-02,
        0.00000000e+00, 2.25890390e-02, 2.59378605e-03, 5.59

In [141]:
px = G.sum(axis=0)

In [159]:
pmi = G / (px[:, None] * px[None, :])
pmi = np.log2(pmi)
np.fill_diagonal(pmi, 0)
pmi

C:\Users\johns\AppData\Local\Temp\ipykernel_14468\960725722.py:2: RuntimeWarning: divide by zero encountered in log2
  pmi = np.log2(pmi)


array([[ 0.        ,  0.58002974,  0.571421  , -0.04117497, -0.04529122,
        -0.10543276, -0.30606346,  0.04674801,  0.25642998,  0.0961636 ,
        -0.08927089, -0.19758182],
       [ 0.58002974,  0.        ,  0.60031162, -0.07808399, -0.07911658,
        -0.12247677, -0.39262329,  0.03606198,  0.21020873,  0.14940171,
        -0.15864464, -0.26480369],
       [ 0.571421  ,  0.60031162,  0.        , -0.06621683, -0.07540889,
        -0.15604149, -0.48272937,  0.07472926,  0.25481086,  0.1515492 ,
        -0.26301211, -0.36934379],
       [-0.04117497, -0.07808399, -0.06621683,  0.        ,  0.59255495,
         0.58806048,  0.55752211, -0.28186304,  0.249061  , -0.21337731,
        -0.01570672, -0.18922929],
       [-0.04529122, -0.07911658, -0.07540889,  0.59255495,  0.        ,
         0.57109567,  0.13158662, -0.32195489,  0.24065771, -0.19151786,
         0.04565065, -0.12764836],
       [-0.10543276, -0.12247677, -0.15604149,  0.58806048,  0.57109567,
         0.        ,  

In [163]:
np.unravel_index(np.nanargmax(pmi), pmi.shape)

(np.int64(10), np.int64(11))

In [164]:
pmi[10, 11]

np.float64(2.836138843313771)

In [165]:
movies.loc[x[10]], movies.loc[x[11]]

(title         Bill & Ted's Excellent Adventure (1989)
 genres                        Adventure|Comedy|Sci-Fi
 imdbId                                          96928
 tmdbId                                         1648.0
 avg_rating                                   6.789852
 num_votes                                        6778
 Name: 4463, dtype: object,
 title         Bill & Ted's Bogus Journey (1991)
 genres          Adventure|Comedy|Fantasy|Sci-Fi
 imdbId                                   101452
 tmdbId                                   1649.0
 avg_rating                             6.047758
 num_votes                                  1951
 Name: 4870, dtype: object)

In [168]:
mutual_info = G * pmi
# mutual_info[np.isnan(mutual_info)] = 0

In [172]:
np.unravel_index(np.nanargmax(mutual_info), mutual_info.shape)

(np.int64(3), np.int64(4))

In [173]:
movies.loc[x[1]], movies.loc[x[2]]

(title         Lord of the Rings: The Two Towers, The (2002)
 genres                                    Adventure|Fantasy
 imdbId                                               167261
 tmdbId                                                121.0
 avg_rating                                         8.161738
 num_votes                                             61601
 Name: 5828, dtype: object,
 title         Lord of the Rings: The Return of the King, The...
 genres                           Action|Adventure|Drama|Fantasy
 imdbId                                                   167260
 tmdbId                                                    122.0
 avg_rating                                             8.220362
 num_votes                                                 63388
 Name: 7011, dtype: object)

In [174]:
marginal_mi = mutual_info.sum(axis=-1)

In [176]:
marginal_mi

array([0.03276878, 0.03044565, 0.03085783, 0.03360944, 0.03041633,
       0.02477255, 0.00211599, 0.00087667, 0.03134488, 0.00207257,
       0.00158135, 0.0011748 ])

In [177]:
marginal_mi.argmax()

np.int64(3)

In [184]:
print("\n".join(movies.loc[x].title.tolist()))

Lord of the Rings: The Fellowship of the Ring, The (2001)
Lord of the Rings: The Two Towers, The (2002)
Lord of the Rings: The Return of the King, The (2003)
Star Wars: Episode IV - A New Hope (1977)
Star Wars: Episode V - The Empire Strikes Back (1980)
Star Wars: Episode VI - Return of the Jedi (1983)
Sense and Sensibility (1995)
Pride and Prejudice (1995)
Matrix, The (1999)
Matrix Revolutions, The (2003)
Bill & Ted's Excellent Adventure (1989)
Bill & Ted's Bogus Journey (1991)


In [185]:
import numpy as np

def get_next_best_item(co_occurrence_matrix, item_labels, steps=3):
    """
    Iteratively finds the item contributing most to Mutual Information,
    then removes it to find the next orthogonal/distinct information hub.
    """
    # Work on a copy to avoid destroying original data
    M = np.array(co_occurrence_matrix, dtype=np.float64)
    
    # Keep track of which indices are still active (not yet removed)
    active_indices = list(range(len(M)))
    results = []

    for step in range(steps):
        # --- Standard MI Calculation on current M ---
        total = np.sum(M)
        if total == 0: break # No data left

        # Probabilities
        P_ij = M / total
        row_sums = np.sum(M, axis=1)
        col_sums = np.sum(M, axis=0)
        
        # Avoid division by zero for removed rows
        with np.errstate(divide='ignore', invalid='ignore'):
            P_i = row_sums / total
            P_j = col_sums / total
            
            # Outer product P(i)*P(j)
            expected = np.outer(P_i, P_j)
            
            # PMI Matrix (log of ratio)
            # We use a mask for P_ij > 0 to handle sparse/zeros safely
            pmi_matrix = np.zeros_like(M)
            nonzero_mask = (P_ij > 0) & (expected > 0)
            
            pmi_matrix[nonzero_mask] = np.log2(P_ij[nonzero_mask] / expected[nonzero_mask])
            
            # Weighted PMI Matrix (P(i,j) * PMI)
            weighted_pmi = P_ij * pmi_matrix

        # --- The "Marginal" Calculation ---
        # Sum the weighted PMI for each row to see that item's contribution
        # Shape: (n,)
        item_scores = np.sum(weighted_pmi, axis=1)
        
        # --- Selection ---
        # We only care about items that haven't been removed yet
        # (Though their rows are already 0, argmax might pick index 0 if all are 0)
        best_idx = np.argmax(item_scores)
        best_score = item_scores[best_idx]
        
        if best_score <= 0:
            print("No more informative items found.")
            break

        results.append((item_labels[best_idx], best_score))
        
        # --- The "Subtraction" / Update Step ---
        # "Explaining away" the item: Remove it from the universe.
        # Set its row and column to 0.
        M[best_idx, :] = 0
        M[:, best_idx] = 0
        
        # Important: The loop continues using the reduced matrix. 
        # The probabilities P(i) will re-adjust to the new Total in the next iter.

    return results

# --- Example Data ---
# 0: Peanut Butter, 1: Jelly, 2: Bread, 3: Tires, 4: Asphalt
labels = ["PB", "Jelly", "Bread", "Tires", "Asphalt"]

# A matrix where (PB, Jelly, Bread) form a cluster, and (Tires, Asphalt) form a cluster
data = [
    [50, 45, 40,  0,  0], # PB
    [45, 50, 40,  0,  0], # Jelly
    [40, 40, 50,  1,  0], # Bread
    [ 0,  0,  1, 50, 45], # Tires
    [ 0,  0,  0, 45, 50]  # Asphalt
]

top_items = get_next_best_item(data, labels, steps=3)

print("--- Optimal Exploration Order ---")
for i, (name, score) in enumerate(top_items):
    print(f"Step {i+1}: Query '{name}' (Score: {score:.4f})")

--- Optimal Exploration Order ---
Step 1: Query 'Asphalt' (Score: 0.2623)
Step 2: Query 'Tires' (Score: 0.3364)
Step 3: Query 'Bread' (Score: 0.0037)


In [194]:
import numpy as np

def get_next_best_item(data, labels, steps=3):
    """
    Identifies the optimal item sequence to maximize information gain (entropy).
    
    This function implements a Greedy Determinantal Point Process (DPP).
    At each step, it selects the item that maximizes the determinant of the 
    sub-matrix of selected items. This maximizes the volume of the feature 
    space covered, ensuring high diversity and information content.
    """
    # Convert data to a numpy array for efficient matrix operations
    K = np.array(data)
    n_items = len(K)
    
    selected_indices = []
    results = []

    for step in range(steps):
        best_score = -float('inf')
        best_idx = -1
        
        # Iterate over all items not yet selected
        for idx in range(n_items):
            if idx in selected_indices:
                continue
            
            # Form a temporary subset including the current candidate
            trial_indices = selected_indices + [idx]
            
            # Extract the submatrix (covariance matrix) for this subset
            # data[np.ix_(rows, cols)] creates the subgrid intersection
            submatrix = K[np.ix_(trial_indices, trial_indices)]
            
            # Calculate the determinant (The "Volume" or "Entropy" score)
            # For a 1x1 matrix, this is just the variance (diagonal value).
            # For larger matrices, this penalizes correlated (redundant) items.
            score = np.linalg.det(submatrix)
            
            # Greedy maximization: Keep the item that results in the highest determinant
            if score > best_score:
                best_score = score
                best_idx = idx
        
        # Lock in the best candidate for this step
        if best_idx != -1:
            selected_indices.append(best_idx)
            results.append((labels[best_idx], best_score))
        else:
            break

    return results

G = mat[:, x].T @ mat[:, x]
G = G.toarray().astype(np.float64)
# np.fill_diagonal(G, 0)
G /= G.sum()

G

top_items = get_next_best_item(G, movies.loc[x, "title"].tolist(), steps=10)

print("--- Optimal Exploration Order ---")
for i, (name, score) in enumerate(top_items):
    print(f"Step {i+1}: Query '{name}' (Score: {score:.4f})")

--- Optimal Exploration Order ---
Step 1: Query 'Matrix, The (1999)' (Score: 0.0353)
Step 2: Query 'Star Wars: Episode IV - A New Hope (1977)' (Score: 0.0008)
Step 3: Query 'Lord of the Rings: The Fellowship of the Ring, The (2001)' (Score: 0.0000)
Step 4: Query 'Star Wars: Episode VI - Return of the Jedi (1983)' (Score: 0.0000)
Step 5: Query 'Star Wars: Episode V - The Empire Strikes Back (1980)' (Score: 0.0000)
Step 6: Query 'Lord of the Rings: The Return of the King, The (2003)' (Score: 0.0000)
Step 7: Query 'Sense and Sensibility (1995)' (Score: 0.0000)
Step 8: Query 'Lord of the Rings: The Two Towers, The (2002)' (Score: 0.0000)
Step 9: Query 'Matrix Revolutions, The (2003)' (Score: 0.0000)
Step 10: Query 'Bill & Ted's Excellent Adventure (1989)' (Score: 0.0000)


In [199]:
m

['Lord of the Rings: The Fellowship of the Ring, The (2001)',
 'Lord of the Rings: The Two Towers, The (2002)',
 'Lord of the Rings: The Return of the King, The (2003)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Star Wars: Episode V - The Empire Strikes Back (1980)',
 'Star Wars: Episode VI - Return of the Jedi (1983)',
 'Sense and Sensibility (1995)',
 'Pride and Prejudice (1995)',
 'Matrix, The (1999)',
 'Matrix Revolutions, The (2003)',
 "Bill & Ted's Excellent Adventure (1989)",
 "Bill & Ted's Bogus Journey (1991)"]

In [220]:
import numpy as np
from scipy.stats import entropy

def estimate_conditional_probs(movie_idx, queried_indices, queried_values, C):
    """Estimate P(other movies | queried movies and current movie)"""
    n = C.shape[0]
    
    # Start with marginal probabilities from diagonal
    probs = np.diag(C) / np.diag(C).sum()
    
    # Update based on queried movies
    for idx, value in zip(queried_indices, queried_values):
        if value == 1:
            # User watched this movie - use co-occurrence
            probs *= (C[idx, :] + 1) / (C[idx, idx] + n)
        else:
            # User didn't watch - inverse relationship
            probs *= 1 - (C[idx, :] + 1) / (C[idx, idx] + n)
    
    # Normalize
    probs = np.clip(probs, 1e-10, 1 - 1e-10)
    return probs / probs.sum()

def compute_expected_entropy(candidate_idx, queried_indices, queried_values, unqueried_mask, C):
    """Compute expected entropy if we query candidate_idx"""
    
    # Probability user watched candidate given what we know
    probs = estimate_conditional_probs(candidate_idx, queried_indices, queried_values, C)
    p_candidate = probs[candidate_idx]
    
    # Entropy if user watched candidate
    probs_if_yes = estimate_conditional_probs(candidate_idx, 
                                               queried_indices + [candidate_idx],
                                               queried_values + [1], C)
    h_if_yes = entropy(probs_if_yes[unqueried_mask])
    
    # Entropy if user didn't watch candidate
    probs_if_no = estimate_conditional_probs(candidate_idx,
                                              queried_indices + [candidate_idx], 
                                              queried_values + [0], C)
    h_if_no = entropy(probs_if_no[unqueried_mask])
    
    return p_candidate * h_if_yes + (1 - p_candidate) * h_if_no

def interactive_query(movies, C, max_queries=5):
    """Interactive movie querying to minimize uncertainty"""
    n = len(movies)
    queried_indices = []
    queried_values = []
    unqueried_mask = np.ones(n, dtype=bool)
    
    print("Answer yes/no to minimize uncertainty about your preferences\n")
    
    for step in range(max_queries):
        if not unqueried_mask.any():
            break
            
        # Find optimal next query
        best_idx = None
        min_entropy = float('inf')
        
        for idx in np.where(unqueried_mask)[0]:
            temp_mask = unqueried_mask.copy()
            temp_mask[idx] = False
            
            exp_ent = compute_expected_entropy(idx, queried_indices, queried_values, 
                                                temp_mask, C)
            
            if exp_ent < min_entropy:
                min_entropy = exp_ent
                best_idx = idx
        
        # Ask the question
        print(f"Q{step+1}: Have you watched '{movies[best_idx]}'?")
        response = input("(yes/no): ").strip().lower()
        
        value = 1 if response in ['yes', 'y'] else 0
        queried_indices.append(best_idx)
        queried_values.append(value)
        unqueried_mask[best_idx] = False
        
        print(f"  → Remaining uncertainty: {min_entropy:.3f}\n")
    
    # Final predictions
    print("\n=== PREDICTIONS ===")
    final_probs = estimate_conditional_probs(None, queried_indices, queried_values, C)
    
    for idx in np.where(unqueried_mask)[0]:
        prob = final_probs[idx]
        prediction = "LIKELY" if prob > 0.5 else "UNLIKELY"
        print(f"{prediction:8s} ({prob:.1%}): {movies[idx]}")

interactive_query(movies.loc[top_k_movies, "title"].tolist(), G, max_queries=5)

Answer yes/no to minimize uncertainty about your preferences

Q1: Have you watched 'Good Will Hunting (1997)'?
(yes/no): yes
  → Remaining uncertainty: 4.530

Q2: Have you watched 'Apollo 13 (1995)'?
(yes/no): no
  → Remaining uncertainty: 4.519

Q3: Have you watched 'Fugitive, The (1993)'?
(yes/no): yes
  → Remaining uncertainty: 4.508

Q4: Have you watched 'Gladiator (2000)'?
(yes/no): no
  → Remaining uncertainty: 4.497

Q5: Have you watched 'Memento (2000)'?
(yes/no): no
  → Remaining uncertainty: 4.486


=== PREDICTIONS ===
UNLIKELY (2.5%): Shawshank Redemption, The (1994)
UNLIKELY (2.1%): Forrest Gump (1994)
UNLIKELY (2.1%): Matrix, The (1999)
UNLIKELY (2.1%): Pulp Fiction (1994)
UNLIKELY (2.0%): Silence of the Lambs, The (1991)
UNLIKELY (1.8%): Star Wars: Episode IV - A New Hope (1977)
UNLIKELY (1.7%): Fight Club (1999)
UNLIKELY (1.7%): Schindler's List (1993)
UNLIKELY (1.5%): Lord of the Rings: The Fellowship of the Ring, The (2001)
UNLIKELY (1.5%): Star Wars: Episode V - The E

In [223]:
top_k_movies = movies.sort_values("num_votes", ascending=False).head(1000).index.tolist()

x = top_k_movies 

G = mat[:, x].T @ mat[:, x]
G = G.toarray().astype(np.float64)
# np.fill_diagonal(G, 0)
G /= G.sum()

G

array([[5.55451156e-05, 2.85510790e-05, 2.73339718e-05, ...,
        1.47653146e-06, 1.47851938e-06, 1.45367032e-06],
       [2.85510790e-05, 4.54872080e-05, 2.27597559e-05, ...,
        1.56300621e-06, 1.13063247e-06, 1.03173319e-06],
       [2.73339718e-05, 2.27597559e-05, 4.53073007e-05, ...,
        1.75931382e-06, 1.19822193e-06, 1.39900238e-06],
       ...,
       [1.47653146e-06, 1.56300621e-06, 1.75931382e-06, ...,
        2.55348995e-06, 8.34928594e-08, 9.84022985e-08],
       [1.47851938e-06, 1.13063247e-06, 1.19822193e-06, ...,
        8.34928594e-08, 2.55348995e-06, 7.20622893e-07],
       [1.45367032e-06, 1.03173319e-06, 1.39900238e-06, ...,
        9.84022985e-08, 7.20622893e-07, 2.54951410e-06]],
      shape=(1000, 1000))

In [226]:
import numpy as np

def get_next_best_item(data, labels, steps=3):
    """
    Identifies the optimal item sequence to maximize information gain (entropy).
    
    This function implements a Greedy Determinantal Point Process (DPP).
    At each step, it selects the item that maximizes the determinant of the 
    sub-matrix of selected items. This maximizes the volume of the feature 
    space covered, ensuring high diversity and information content.
    """
    # Convert data to a numpy array for efficient matrix operations
    K = np.array(data)
    n_items = len(K)
    
    selected_indices = []
    results = []

    for step in range(steps):
        best_score = -float('inf')
        best_idx = -1
        
        # Iterate over all items not yet selected
        for idx in range(n_items):
            if idx in selected_indices:
                continue
            
            # Form a temporary subset including the current candidate
            trial_indices = selected_indices + [idx]
            
            # Extract the submatrix (covariance matrix) for this subset
            # data[np.ix_(rows, cols)] creates the subgrid intersection
            submatrix = K[np.ix_(trial_indices, trial_indices)]
            
            # Calculate the determinant (The "Volume" or "Entropy" score)
            # For a 1x1 matrix, this is just the variance (diagonal value).
            # For larger matrices, this penalizes correlated (redundant) items.
            score = np.linalg.det(submatrix)
            
            # Greedy maximization: Keep the item that results in the highest determinant
            if score > best_score:
                best_score = score
                best_idx = idx
        
        # Lock in the best candidate for this step
        if best_idx != -1:
            selected_indices.append(best_idx)
            results.append((labels[best_idx], best_score))
        else:
            break

    return results

top_items = get_next_best_item(G, movies.loc[top_k_movies, "title"].tolist(), steps=40)

print("--- Optimal Exploration Order ---")
for i, (name, score) in enumerate(top_items):
    print(f"Step {i+1}: Query '{name}' (Score: {score:.4f})")

--- Optimal Exploration Order ---
Step 1: Query 'Shawshank Redemption, The (1994)' (Score: 0.0001)
Step 2: Query 'Matrix, The (1999)' (Score: 0.0000)
Step 3: Query 'Forrest Gump (1994)' (Score: 0.0000)
Step 4: Query 'Pulp Fiction (1994)' (Score: 0.0000)
Step 5: Query 'Star Wars: Episode IV - A New Hope (1977)' (Score: 0.0000)
Step 6: Query 'Silence of the Lambs, The (1991)' (Score: 0.0000)
Step 7: Query 'Schindler's List (1993)' (Score: 0.0000)
Step 8: Query 'Godfather, The (1972)' (Score: 0.0000)
Step 9: Query 'Toy Story (1995)' (Score: 0.0000)
Step 10: Query 'Fight Club (1999)' (Score: 0.0000)
Step 11: Query 'Braveheart (1995)' (Score: 0.0000)
Step 12: Query 'Lord of the Rings: The Fellowship of the Ring, The (2001)' (Score: 0.0000)
Step 13: Query 'American Beauty (1999)' (Score: 0.0000)
Step 14: Query 'Inception (2010)' (Score: 0.0000)
Step 15: Query 'Usual Suspects, The (1995)' (Score: 0.0000)
Step 16: Query 'Jurassic Park (1993)' (Score: 0.0000)
Step 17: Query 'Raiders of the Lost

In [213]:
# movies.loc[x, "title"].tolist()

In [214]:
import numpy as np
from scipy.stats import entropy

def estimate_conditional_probs(movie_idx, queried_indices, queried_values, C):
    """Estimate P(other movies | queried movies and current movie)"""
    n = C.shape[0]
    
    # Start with marginal probabilities from diagonal
    probs = np.diag(C) / np.diag(C).sum()
    
    # Update based on queried movies
    for idx, value in zip(queried_indices, queried_values):
        if value == 1:
            # User watched this movie - use co-occurrence
            probs *= (C[idx, :] + 1) / (C[idx, idx] + n)
        else:
            # User didn't watch - inverse relationship
            probs *= 1 - (C[idx, :] + 1) / (C[idx, idx] + n)
    
    # Normalize
    probs = np.clip(probs, 1e-10, 1 - 1e-10)
    return probs / probs.sum()

def compute_expected_entropy(candidate_idx, queried_indices, queried_values, unqueried_mask, C):
    """Compute expected entropy if we query candidate_idx"""
    
    # Probability user watched candidate given what we know
    probs = estimate_conditional_probs(candidate_idx, queried_indices, queried_values, C)
    p_candidate = probs[candidate_idx]
    
    # Entropy if user watched candidate
    probs_if_yes = estimate_conditional_probs(candidate_idx, 
                                               queried_indices + [candidate_idx],
                                               queried_values + [1], C)
    h_if_yes = entropy(probs_if_yes[unqueried_mask])
    
    # Entropy if user didn't watch candidate
    probs_if_no = estimate_conditional_probs(candidate_idx,
                                              queried_indices + [candidate_idx], 
                                              queried_values + [0], C)
    h_if_no = entropy(probs_if_no[unqueried_mask])
    
    return p_candidate * h_if_yes + (1 - p_candidate) * h_if_no

def interactive_query(movies, C, max_queries=5):
    """Interactive movie querying to minimize uncertainty"""
    n = len(movies)
    queried_indices = []
    queried_values = []
    unqueried_mask = np.ones(n, dtype=bool)
    
    print("Answer yes/no to minimize uncertainty about your preferences\n")
    
    for step in range(max_queries):
        if not unqueried_mask.any():
            break
            
        # Find optimal next query
        best_idx = None
        min_entropy = float('inf')
        
        for idx in np.where(unqueried_mask)[0]:
            temp_mask = unqueried_mask.copy()
            temp_mask[idx] = False
            
            exp_ent = compute_expected_entropy(idx, queried_indices, queried_values, 
                                                temp_mask, C)
            
            if exp_ent < min_entropy:
                min_entropy = exp_ent
                best_idx = idx
        
        # Ask the question
        print(f"Q{step+1}: Have you watched '{movies[best_idx]}'?")
        response = input("(yes/no): ").strip().lower()
        
        value = 1 if response in ['yes', 'y'] else 0
        queried_indices.append(best_idx)
        queried_values.append(value)
        unqueried_mask[best_idx] = False
        
        print(f"  → Remaining uncertainty: {min_entropy:.3f}\n")
    
    # Final predictions
    print("\n=== PREDICTIONS ===")
    final_probs = estimate_conditional_probs(None, queried_indices, queried_values, C)
    
    for idx in np.where(unqueried_mask)[0]:
        prob = final_probs[idx]
        prediction = "LIKELY" if prob > 0.5 else "UNLIKELY"
        print(f"{prediction:8s} ({prob:.1%}): {movies[idx]}")

# Example co-occurrence matrix (symmetric, diagonal = view counts)
# movies = ['Lord of the Rings: The Fellowship of the Ring, The (2001)',
#           'Lord of the Rings: The Two Towers, The (2002)',
#           'Lord of the Rings: The Return of the King, The (2003)',
#           'Star Wars: Episode IV - A New Hope (1977)',
#           'Star Wars: Episode V - The Empire Strikes Back (1980)',
#           'Star Wars: Episode VI - Return of the Jedi (1983)',
#           'Sense and Sensibility (1995)',
#           'Pride and Prejudice (1995)',
#           'Matrix, The (1999)',
#           'Matrix Revolutions, The (2003)',
#           "Bill & Ted's Excellent Adventure (1989)",
#           "Bill & Ted's Bogus Journey (1991)"]

# Synthetic co-occurrence matrix (replace with your actual data)
# C = np.array([
#     [1000, 950, 940, 600, 580, 570, 200, 190, 500, 480, 300, 290],
#     [950, 1000, 980, 590, 570, 560, 195, 185, 490, 470, 295, 285],
#     [940, 980, 1000, 580, 560, 550, 190, 180, 485, 465, 290, 280],
#     [600, 590, 580, 1000, 920, 910, 250, 240, 550, 530, 400, 390],
#     [580, 570, 560, 920, 1000, 950, 245, 235, 540, 520, 395, 385],
#     [570, 560, 550, 910, 950, 1000, 240, 230, 535, 515, 390, 380],
#     [200, 195, 190, 250, 245, 240, 1000, 880, 220, 210, 150, 145],
#     [190, 185, 180, 240, 235, 230, 880, 1000, 215, 205, 145, 140],
#     [500, 490, 485, 550, 540, 535, 220, 215, 1000, 850, 450, 440],
#     [480, 470, 465, 530, 520, 515, 210, 205, 850, 1000, 440, 430],
#     [300, 295, 290, 400, 395, 390, 150, 145, 450, 440, 1000, 780],
#     [290, 285, 280, 390, 385, 380, 145, 140, 440, 430, 780, 1000]
# ])

# C = np.copy(G)

C = np.array([
    [50, 45, 40,  0,  0], # PB
    [45, 50, 40,  0,  0], # Jelly
    [40, 40, 50,  1,  0], # Bread
    [ 0,  0,  1, 50, 45], # Tires
    [ 0,  0,  0, 45, 50]  # Asphalt
])

movies = ["PB", "Jelly", "Bread", "Tires", "Asphault"]

interactive_query(movies, C, max_queries=5)

Answer yes/no to minimize uncertainty about your preferences

Q1: Have you watched 'Asphault'?
(yes/no): exi
  → Remaining uncertainty: 1.057

Q2: Have you watched 'PB'?
(yes/no): no
  → Remaining uncertainty: 0.961

Q3: Have you watched 'Bread'?
(yes/no): no
  → Remaining uncertainty: 0.399

Q4: Have you watched 'Jelly'?
(yes/no): no
  → Remaining uncertainty: 0.000

Q5: Have you watched 'Tires'?
(yes/no): no
  → Remaining uncertainty: 0.000


=== PREDICTIONS ===


In [192]:
G

array([[0.00000000e+00, 2.64267288e-02, 2.63827746e-02, 1.81804330e-02,
        1.72671624e-02, 1.47124468e-02, 1.99356706e-03, 7.51616782e-04,
        2.29816965e-02, 4.67672664e-03, 2.17426765e-03, 6.43684807e-04],
       [2.64267288e-02, 0.00000000e+00, 2.56541117e-02, 1.68901331e-02,
        1.60760036e-02, 1.38577818e-02, 1.78942423e-03, 7.11081244e-04,
        2.12132726e-02, 4.62495837e-03, 1.97500862e-03, 5.85567590e-04],
       [2.63827746e-02, 2.56541117e-02, 0.00000000e+00, 1.71030667e-02,
        1.61868659e-02, 1.35974753e-02, 1.68832957e-03, 7.33546723e-04,
        2.19736802e-02, 4.65181926e-03, 1.84509955e-03, 5.46985572e-04],
       [1.81804330e-02, 1.68901331e-02, 1.71030667e-02, 0.00000000e+00,
        2.70977629e-02, 2.39965501e-02, 3.65845439e-03, 6.03637649e-04,
        2.30603257e-02, 3.80594515e-03, 2.30759538e-03, 6.52964027e-04],
       [1.72671624e-02, 1.60760036e-02, 1.61868659e-02, 2.70977629e-02,
        0.00000000e+00, 2.25890390e-02, 2.59378605e-03, 5.59

In [189]:
movies.loc[x, "title"].tolist()

['Lord of the Rings: The Fellowship of the Ring, The (2001)',
 'Lord of the Rings: The Two Towers, The (2002)',
 'Lord of the Rings: The Return of the King, The (2003)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Star Wars: Episode V - The Empire Strikes Back (1980)',
 'Star Wars: Episode VI - Return of the Jedi (1983)',
 'Sense and Sensibility (1995)',
 'Pride and Prejudice (1995)',
 'Matrix, The (1999)',
 'Matrix Revolutions, The (2003)',
 "Bill & Ted's Excellent Adventure (1989)",
 "Bill & Ted's Bogus Journey (1991)"]

In [139]:
mutual_info

array([[            nan,  1.06247600e-02,  1.04496592e-02,
        -5.18875304e-04, -5.42076305e-04, -1.07519181e-03,
        -4.22929315e-04,  2.43548290e-05,  4.08485221e-03,
         3.11729671e-04, -1.34539040e-04, -8.81547459e-05],
       [ 1.06247600e-02,             nan,  1.06747864e-02,
        -9.14156497e-04, -8.81598978e-04, -1.17644844e-03,
        -4.86984162e-04,  1.77743708e-05,  3.09089234e-03,
         4.78948547e-04, -2.17180019e-04, -1.07479719e-04],
       [ 1.04496592e-02,  1.06747864e-02,             nan,
        -7.84996684e-04, -8.46078738e-04, -1.47069908e-03,
        -5.64919299e-04,  3.79965287e-05,  3.88102287e-03,
         4.88654550e-04, -3.36372912e-04, -1.40033562e-04],
       [-5.18875304e-04, -9.14156497e-04, -7.84996684e-04,
                    nan,  1.11298044e-02,  9.78129298e-03,
         1.41379098e-03, -1.17934241e-04,  3.98104078e-03,
        -5.62906442e-04, -2.51229501e-05, -8.56452104e-05],
       [-5.42076305e-04, -8.81598978e-04, -8.4607873

In [103]:
mutual_info[1, 2]

np.float64(0.0045924743949344055)

In [92]:
G[3, 4]

np.float64(0.027097762926685375)

In [77]:
np.unravel_index(np.nanargmax(G), mutual_info.shape)

(np.int64(0), np.int64(3))

In [56]:
np.log(G)

array([[ -3.65426368,  -3.86431531,  -3.86597994,  -4.23834536,
         -4.28988469,  -4.44999741,  -6.44876573,  -7.42421994,
         -4.00399317,  -5.59609283,  -6.36199937,  -7.57923736],
       [ -3.86431531,  -3.73467469,  -3.8939874 ,  -4.31196165,
         -4.36136356,  -4.50984432,  -6.55679735,  -7.47965985,
         -4.08406421,  -5.60722389,  -6.4581185 ,  -7.67386492],
       [ -3.86597994,  -3.8939874 ,  -3.70607823,  -4.29943347,
         -4.35449109,  -4.52880713,  -6.61495164,  -7.44855525,
         -4.04884588,  -5.60143288,  -6.52615803,  -7.74202411],
       [ -4.23834536,  -4.31196165,  -4.29943347,  -3.4821525 ,
         -3.83924009,  -3.96078119,  -5.8416505 ,  -7.64347244,
         -4.00057762,  -5.8021269 ,  -6.30248524,  -7.5649245 ],
       [ -4.28988469,  -4.36136356,  -4.35449109,  -3.83924009,
         -3.6568134 ,  -4.02122647,  -6.18557266,  -7.71994816,
         -4.05508851,  -5.83566126,  -6.30864172,  -7.57092602],
       [ -4.44999741,  -4.50984432,

In [39]:
mat_small = mat[:, x][x, :]

In [42]:
mat_small.toarray()

array([[0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0],
       [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0],
       [1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [43]:
mat

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 21381913 stored elements and shape (326680, 64548)>

In [9]:
# lambda_ = optimize_lambda_using_a_to_b_matching(mat, a, b, fast_approximation=False)
lambda_ = optimize_lambda_using_a_to_b_matching(mat, a, b, fast_approximation=True)

lambda_: 1000000
error: 182.999999
lambda_: 100000.0
error: 42.99999
lambda_: 10000.0
error: 11.9999
lambda_: 1000.0
error: 9.999
lambda_: 100.0
error: 13.99


In [10]:
# check error for EASE with optimized lambda_
a_to_b_error_metric(mat, a, b, lambda_, lambda_penalty=False)

lambda_: 1000
error: 9


9

In [11]:
# check error for NPMI in comparison
a_to_b_error_metric_npmi(mat, a, b, temp=1)

temp: 1
error: 1782.0


1782.0

In [12]:
top_k = 20

In [13]:
# using EASE

similarity_scores = calculate_ease_for_item_cg(mat, a, lambda_)

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

,title,genres,imdbId,tmdbId,avg_rating,num_votes
movieId,,,,,,
819,Emma (1996),Comedy|Drama|Romance,116191,3573.0,7.485110,6486
27,Persuasion (1995),Drama|Romance,114117,17015.0,8.076637,2719
7382,Pride and Prejudice (1995),Drama|Romance,112130,164721.0,7.989470,2607
510,"Remains of the Day, The (1993)",Drama|Romance,107943,1245.0,7.783682,8651
605,Jane Eyre (1996),Drama|Romance,116684,47333.0,7.271146,1807
57,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,110877,11010.0,7.927828,10200
10352,Pride & Prejudice (2005),Drama|Romance,414387,4348.0,7.703508,6645
492,Much Ado About Nothing (1993),Comedy|Romance,107616,11971.0,7.734885,11266
258,Little Women (1994),Drama,110367,9587.0,7.199093,7447


In [14]:
# using EASE

similarity_scores = calculate_ease_for_item_cg(mat, b, lambda_)

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

,title,genres,imdbId,tmdbId,avg_rating,num_votes
movieId,,,,,,
14199,Persuasion (2007),Drama|Romance,844330,13949.0,7.719715,330
17597,North & South (2004),Drama|Romance,417349,147269.0,8.057732,400
10352,Pride & Prejudice (2005),Drama|Romance,414387,4348.0,7.703508,6645
13158,"Young Victoria, The (2009)",Drama|Romance,962736,18320.0,7.472767,679
27,Persuasion (1995),Drama|Romance,114117,17015.0,8.076637,2719
15923,Jane Eyre (2011),Drama|Romance,1229822,38684.0,7.451439,824
16728,Northanger Abbey (2007),Drama|Romance,844794,18093.0,7.429658,201
11401,Becoming Jane (2007),Drama|Romance,416508,2977.0,7.129006,837
2984,Mansfield Park (1999),Comedy|Drama|Romance,178737,10399.0,7.523058,1191


In [15]:
# using normalized pointwise mutual information

similarity_scores = npmi_batch(mat, a)

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

,title,genres,imdbId,tmdbId,avg_rating,num_votes
movieId,,,,,,
819,Emma (1996),Comedy|Drama|Romance,116191,3573.0,7.485110,6486
27,Persuasion (1995),Drama|Romance,114117,17015.0,8.076637,2719
510,"Remains of the Day, The (1993)",Drama|Romance,107943,1245.0,7.783682,8651
492,Much Ado About Nothing (1993),Comedy|Romance,107616,11971.0,7.734885,11266
262,Like Water for Chocolate (Como agua para choco...,Drama|Fantasy|Romance,103994,18183.0,7.827167,9183
57,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,110877,11010.0,7.927828,10200
529,Shadowlands (1993),Drama|Romance,108101,10445.0,7.772285,3548
258,Little Women (1994),Drama,110367,9587.0,7.199093,7447
352,Four Weddings and a Funeral (1994),Comedy|Romance,109831,712.0,7.284194,19987


In [16]:
# using normalized pointwise mutual information

similarity_scores = npmi_batch(mat, b)

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

,title,genres,imdbId,tmdbId,avg_rating,num_votes
movieId,,,,,,
14199,Persuasion (2007),Drama|Romance,844330,13949.0,7.719715,330
17597,North & South (2004),Drama|Romance,417349,147269.0,8.057732,400
16728,Northanger Abbey (2007),Drama|Romance,844794,18093.0,7.429658,201
13158,"Young Victoria, The (2009)",Drama|Romance,962736,18320.0,7.472767,679
10352,Pride & Prejudice (2005),Drama|Romance,414387,4348.0,7.703508,6645
11401,Becoming Jane (2007),Drama|Romance,416508,2977.0,7.129006,837
15923,Jane Eyre (2011),Drama|Romance,1229822,38684.0,7.451439,824
24799,Sense & Sensibility (2008),Drama|Romance,847150,315010.0,7.693069,76
20467,Cranford (2007),Drama,974077,64047.0,7.877193,46


In [17]:
# average the two rankings

similarity_scores_a = calculate_ease_for_item_cg(mat, a, lambda_)
similarity_ranking_a = np.full(len(similarity_scores_a), -1.0)
similarity_ranking_a[np.argsort(-similarity_scores_a)] = 1.0 - ((1.0 + np.arange(len(similarity_ranking_a)))/len(similarity_ranking_a))

similarity_scores_b = calculate_ease_for_item_cg(mat, b, lambda_)
similarity_ranking_b = np.full(len(similarity_scores_b), -1.0)
similarity_ranking_b[np.argsort(-similarity_scores_b)] = 1.0 - ((1.0 + np.arange(len(similarity_ranking_b)))/len(similarity_ranking_b))

similarity_scores = similarity_ranking_a * similarity_ranking_b
# similarity_scores = similarity_scores_a * similarity_scores_b

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

for m in movies.loc[np.argsort(-similarity_scores)[:top_k], "title"]:
    print(m)

Persuasion (1995)
Pride & Prejudice (2005)
Emma (1996)
Mansfield Park (1999)
Becoming Jane (2007)
Persuasion (2007)
North & South (2004)
Jane Eyre (2011)
Importance of Being Earnest, The (2002)
Northanger Abbey (2007)
Jane Eyre (1996)
Duchess, The (2008)
Roman Holiday (1953)
Far from the Madding Crowd (2015)
Bridget Jones's Diary (2001)
The Queen (2006)
Fiddler on the Roof (1971)
Pride and Prejudice (1940)
Phantom of the Opera, The (2004)
Room with a View, A (1986)
